# Pruebas — Clasificador de cantos (Módulo 1)

Notebook de experimentación del proyecto de titulación. Usa el kernel **Python (biomonitor)**. Cubre: cargar el dataset cacheado, visualizar/oír espectrogramas, sanity-check del modelo y un mini-entrenamiento de prueba.

> El entrenamiento completo se corre con `scripts/07_train_audio.py`; aquí es solo para probar rápido e iterar.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
from pathlib import Path

# Ubicar la raíz del proyecto (carpeta con pyproject.toml) sin importar desde dónde se abra
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Raíz del proyecto:', ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from biomonitor.config import CONFIG, PROJECT_ROOT, data_path

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('Dispositivo:', device, '| torch', torch.__version__)

## 2. Cargar el dataset cacheado

In [ ]:
manifest = pd.read_csv(data_path('processed', 'audio32_manifest.csv'))
species = pd.read_csv(data_path('external', 'species_final.csv'))['scientific_name'].tolist()
print('grabaciones:', len(manifest), '| especies:', manifest['scientific_name'].nunique())
print('split:', manifest['split'].value_counts().to_dict())
manifest.head()

In [ ]:
# Distribución de grabaciones por especie
counts = manifest.groupby('scientific_name').size().sort_values()
ax = counts.plot(kind='barh', figsize=(6, 12))
ax.set_xlabel('nº grabaciones'); ax.set_title('Grabaciones por especie'); plt.tight_layout()

## 3. Visualizar y oír un espectrograma

In [ ]:
from biomonitor.audio.dataset import BirdSpectrogramDataset, build_label_map
from IPython.display import Audio, display

label_map = build_label_map(species)
ds_vis = BirdSpectrogramDataset(manifest, label_map, train=False)

idx = 0  # cambia el índice para explorar
spec, label = ds_vis[idx]
row = manifest.iloc[idx]

plt.figure(figsize=(9, 3))
plt.imshow(spec.squeeze(0).numpy(), origin='lower', aspect='auto')
plt.title(f"{row['scientific_name']}  (clase {label}, {row['split']})")
plt.xlabel('frames'); plt.ylabel('mel'); plt.colorbar()
plt.show()

display(Audio(str(PROJECT_ROOT / row['cached_file'])))

## 4. Sanity-check del modelo

In [ ]:
from torch.utils.data import DataLoader
from biomonitor.audio.dataset import datasets_from_manifest
from biomonitor.audio.model import build_audio_model

train_ds, val_ds, test_ds, label_map = datasets_from_manifest(manifest, species)
loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
x, y = next(iter(loader))
print('batch:', tuple(x.shape), '| labels:', y.tolist())

model = build_audio_model('efficientnet_b0', num_classes=len(label_map), pretrained=False).to(device)
logits = model(x.to(device))
print('logits:', tuple(logits.shape))

## 5. Mini-entrenamiento de prueba

Unas pocas iteraciones para confirmar que la pérdida baja. **No** es el entrenamiento real (usa `scripts/07_train_audio.py` para eso).

In [ ]:
import torch.nn as nn

model = build_audio_model('efficientnet_b0', num_classes=len(label_map), pretrained=True).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)

model.train()
for step, (x, y) in enumerate(loader):
    x, y = x.to(device), y.to(device)
    logits = model(x); loss = crit(logits, y)
    opt.zero_grad(); loss.backward(); opt.step()
    print(f'step {step:>2}  loss {loss.item():.3f}')
    if step >= 15:
        break